In [8]:
import os
import pickle
import cv2
import numpy as np

# Define global settings and target file paths
ORB_FEATURES = 500
TEST_IMAGE_PATH = "../data/test/clouds.jpg"  
MAP_FILE = "../build/image_map.pkl"

print("Loading offline database structures...")
with open(MAP_FILE, "rb") as f:
    db_data = pickle.load(f)

# Unpack the stored variables from the pickled database file
image_id_map = db_data["image_id_map"]
image_filenames = db_data["image_filenames"]
global_descriptor_matrix = db_data["global_descriptor_matrix"]

print(f"Database loaded: {len(image_filenames)} gallery images, {len(global_descriptor_matrix)} total features")

print("Building LSH search index in memory...")
FLANN_INDEX_LSH = 6
# Configure the Locality-Sensitive Hashing (LSH) parameters for binary descriptors
index_params = dict(algorithm=FLANN_INDEX_LSH,
                    table_number=6,      
                    key_size=12,          
                    multi_probe_level=1)
# Build the searchable tree structures in RAM using our global feature matrix
flann_index = cv2.flann_Index(global_descriptor_matrix, index_params)
print("Index initialized successfully.")

print(f"\nReading query image from: {TEST_IMAGE_PATH}")
query_img = cv2.imread(TEST_IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

if query_img is None:
    print(f"Error: Could not read {TEST_IMAGE_PATH}. Check your filename in data/test/!")
else:
    # Initialize ORB and extract a maximum of 500 features from the incoming test image
    orb = cv2.ORB_create(nfeatures=ORB_FEATURES)
    _, query_descriptors = orb.detectAndCompute(query_img, None)
    
    if query_descriptors is None:
        print("Error: No keypoints could be extracted from the query image.")
    else:
        print(f"Extracted {len(query_descriptors)} features from the query image.")
        
        # Use the LSH tree to find the single 1st nearest neighbor for every query feature
        search_params = dict(checks=50)
        idx, dist = flann_index.knnSearch(query_descriptors, knn=1, params=search_params)
        idx = idx.ravel()
        dist = dist.ravel()
        
        # Prepare an array of empty slots to count votes for each image in the registry
        num_gallery_images = len(image_filenames)
        votes = np.zeros(num_gallery_images, dtype=np.int32)
        
        # Maximum allowed bit-difference for a feature match to be considered valid
        HAMMING_THRESHOLD = 50 
        
        # Iterate over every query feature and cast a vote if it passes the quality threshold
        for i in range(len(query_descriptors)):
            if dist[i] < HAMMING_THRESHOLD:
                global_idx = idx[i]
                matched_image_id = image_id_map[global_idx]
                votes[matched_image_id] += 1
        
        print("\n--- OVERALL DATABASE VOTES ---")
        # Print results only for images that received at least one feature match
        for img_id, name in image_filenames.items():
            if votes[img_id] > 0:
                print(f"Image ID {img_id} ({name}): {votes[img_id]} feature matches")
            
        # Identify the image that received the absolute highest number of votes
        winner_id = np.argmax(votes)
        max_votes = votes[winner_id]
        suspect_filename = image_filenames[winner_id]
        
        print("------------------------------")
        
        # STAGE 1 PASSTHROUGH: Verify if the suspect image passed our calibrated minimum filter
        if max_votes > 5: 
            print("hhhhhhhhh")
            print(f"⚠️ SUSPICION TRIGGERED: Prime suspect identified as '{suspect_filename}' with {max_votes} votes.")
            print("Initiating Stage 2: Targeted Spatial Verification via RANSAC...")
            
            # Read the original source image from disk to perform a direct geometric comparison
            suspect_path = os.path.join("../data/raw", suspect_filename)
            suspect_img = cv2.imread(suspect_path, cv2.IMREAD_GRAYSCALE)
            
            if suspect_img is None:
                print(f"Error: Could not load original suspect file at {suspect_path} for verification.")
            else:
                # Extract clean keypoint coordinates from both the suspect and the query image
                kp_suspect, des_suspect = orb.detectAndCompute(suspect_img, None)
                kp_query, des_query = orb.detectAndCompute(query_img, None)
                
                # Perform a rigorous 1-to-1 brute force cross-check match between these two images
                bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
                matches = bf.match(des_query, des_suspect)
                
                # Reshape coordinates into 2D float matrices for geometric homography calculation
                src_pts = np.float32([kp_query[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
                dst_pts = np.float32([kp_suspect[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
                
                if len(src_pts) >= 4:
                    # Run RANSAC to see if the matching points share a unified spatial transformation
                    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
                    
                    if mask is not None:
                        inlier_matches = np.sum(mask)
                        print(f"RANSAC Results: {inlier_matches} features perfectly align geometrically.")
                        
                        # STAGE 2 PASSTHROUGH: Confirm plagiarism if the spatial alignment crosses our threshold
                        if inlier_matches > 15:
                            print(f"\n🚨🚨 PLAGIARISM CONFIRMED! 🚨🚨")
                            print(f"The uploaded image is a mathematically verified manipulation of '{suspect_filename}'.")
                            print(f"Minting blocked by off-chain verification network.")
                        else:
                            print("\n✅ FALSE ALARM: Random texture similarities detected, but geometric structures do not match.")
                    else:
                        print("\n✅ FALSE ALARM: No geometric alignment possible.")
                else:
                    print("\n✅ FALSE ALARM: Not enough matching points for geometry check.")
        else:
            print("✅ CLEAN: No matching imagery profiles found in the registry.")

Loading offline database structures...
Database loaded: 202 gallery images, 61630 total features
Building LSH search index in memory...
Index initialized successfully.

Reading query image from: ../data/test/clouds.jpg
Extracted 500 features from the query image.

--- OVERALL DATABASE VOTES ---
Image ID 0 (cryptopunk_119.png): 1 feature matches
Image ID 1 (cryptopunk_59.png): 2 feature matches
Image ID 2 (cryptopunk_39.png): 1 feature matches
Image ID 4 (cryptopunk_107.png): 1 feature matches
Image ID 6 (cryptopunk_62.png): 1 feature matches
Image ID 8 (cryptopunk_77.png): 1 feature matches
Image ID 9 (cryptopunk_146.png): 2 feature matches
Image ID 15 (sky2.jpg): 35 feature matches
Image ID 19 (cryptopunk_187.png): 1 feature matches
Image ID 22 (cryptopunk_159.png): 2 feature matches
Image ID 25 (cryptopunk_179.png): 1 feature matches
Image ID 26 (cryptopunk_194.png): 1 feature matches
Image ID 28 (cryptopunk_181.png): 5 feature matches
Image ID 32 (cryptopunk_147.png): 3 feature matc